# Swiss People Data Simulator

Generates a fake dataset of Swiss people (names, addresses, phone numbers) with [Faker](https://faker.readthedocs.io) and saves it as `swiss_people.csv`.

Faker supports Swiss locales: `de_CH` (German), `fr_CH` (French) and `it_CH` (Italian). Note: `it_CH` has no Swiss address provider (it falls back to US addresses), so Italian-speaking people get Italian names but addresses from `de_CH`.

All data is fake — postcodes, cities and cantons are not necessarily consistent with each other.

## Libraries and settings

In [1]:
# Libraries
import os
import numpy as np
import pandas as pd
from faker import Faker

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

/workspaces/data_analytics/Week_LC/LC_06


## Random seed
Seed both Faker and NumPy, so the simulated data is reproducible.

In [2]:
seed = 42
n_people = 1000  # number of people to generate

Faker.seed(seed)
rng = np.random.default_rng(seed)

# One Faker instance per Swiss language region
fake = {
    'de': Faker('de_CH'),
    'fr': Faker('fr_CH'),
    'it': Faker('it_CH'),
}

# Approximate share of language regions in Switzerland
languages = ['de', 'fr', 'it']
lang_probs = [0.65, 0.27, 0.08]

## Generate the dataset

In [3]:
def make_person(lang):
    f = fake[lang]
    f_addr = fake['fr'] if lang == 'fr' else fake['de']  # it_CH has no Swiss addresses
    gender = rng.choice(['female', 'male'])
    first_name = f.first_name_female() if gender == 'female' else f.first_name_male()
    last_name = f.last_name()
    canton_code, canton_name = fake['de'].canton()
    return {
        'first_name': first_name,
        'last_name': last_name,
        'gender': gender,
        'language': lang,
        'birth_date': f.date_of_birth(minimum_age=18, maximum_age=90),
        'street': f_addr.street_address(),
        'postcode': f_addr.postcode(),
        'city': f_addr.city(),
        'canton': canton_code,
        'canton_name': canton_name,
        'phone': f.phone_number(),
        'mobile': '+41 ' + rng.choice(['76', '77', '78', '79']) + f' {rng.integers(100, 1000)} {rng.integers(10, 100)} {rng.integers(10, 100)}',
        'email': f'{first_name}.{last_name}@{f.free_email_domain()}'.lower().replace(' ', ''),
    }

langs = rng.choice(languages, size=n_people, p=lang_probs)
df = pd.DataFrame([make_person(lang) for lang in langs])
df.insert(0, 'person_id', range(1, n_people + 1))

print(df.shape)
df.head(10)

(1000, 14)


,person_id,first_name,last_name,gender,language,birth_date,street,postcode,city,canton,canton_name,phone,mobile,email
0,1,Eric,Bahon,male,fr,1955-11-17,rue Rappaz 7,1013,Pachoud,VS,Wallis,050 838 63 79,+41 76 457 51 81,eric.bahon@romandie.com
1,2,Ruzica,Aebi,female,de,1994-08-17,Herzogstrasse 25,2615,Wohlen,ZG,Zug,022 781 61 84,+41 77 237 80 66,ruzica.aebi@hotmail.com
2,3,Madeleine,Rossellat,female,fr,1987-03-27,rue de Deladoëy,2316,Monney am Rhein,FR,Freiburg,+41 75 534 19 28,+41 77 237 92 68,madeleine.rossellat@bluewin.ch
3,4,Eliane,Morand,female,fr,1955-07-04,boulevard de Comment 40,1564,Coigny,NE,Neuenburg,043 537 67 24,+41 77 753 74 51,eliane.morand@bluewin.ch
4,5,Malgorzata,Schaub,female,de,1954-12-24,Kälinstrasse 88,2012,Regensdorf,SO,Solothurn,+41 52 697 84 80,+41 78 692 63 16,malgorzata.schaub@hotmail.com
5,6,Michelle,Maldonado,female,it,1990-08-13,Hoferstrasse 17,5828,Zug,SO,Solothurn,+41 (0)39 893 25 28,+41 78 955 57 40,michelle.maldonado@gmail.com
6,7,Marcel,Niquille,male,fr,1943-12-19,rue Eric Cornut 811,2822,Mottiez (SO),AG,Aargau,+41 74 896 38 34,+41 76 181 71 98,marcel.niquille@sunrise.ch
7,8,Michel,Meyer,male,fr,1968-10-04,rue Galland 9,4930,Wicht,SH,Schaffhausen,+41 33 105 18 34,+41 78 136 98 50,michel.meyer@vtxnet.ch
8,9,Erna,Rüegg,female,de,1988-08-09,Wüthrichstrasse 23,2656,Schlieren,BL,Basel-Landschaft,+41 26 065 13 33,+41 79 771 77 34,erna.rüegg@hotmail.com
9,10,Carmine,Eichenberger,male,de,1949-03-12,Bianchistrasse 910,1132,Spiez,NW,Nidwalden,032 602 60 64,+41 77 578 77 69,carmine.eichenberger@yahoo.com


## Quick overview

In [4]:
print(df['language'].value_counts(), '\n')
print(df['gender'].value_counts(), '\n')
print(df['canton'].value_counts().head(10))

language
de    649
fr    273
it     78
Name: count, dtype: int64 

gender
female    526
male      474
Name: count, dtype: int64 

canton
ZG    53
BE    51
LU    49
OW    49
JU    45
SO    44
ZH    43
NE    42
NW    41
TI    40
Name: count, dtype: int64


## Save to CSV

In [5]:
df.to_csv('swiss_people.csv', index=False)
print('Saved swiss_people.csv')

Saved swiss_people.csv


### Jupyter notebook --footer info--

In [6]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-20 13:10:38
Python Version: 3.11.16
-----------------------------------
